# 07 時間序列與預測：從滾動平均到 ARIMA/SARIMA

松柏護理之家退伍軍人症群聚事件進入第二週，長官丟出兩個問題：
> 1. 「下禮拜還會有多少人發病？醫院還要準備幾張床？」
> 2. 「**明天**會不會又是一個高峰日？要不要提前啟動警報？」

這本 notebook 用**六種模型**回答這兩個問題：rolling mean、Poisson+lag、Negative Binomial+lag、Logistic、ARIMA、SARIMA。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: 建立每日發病數序列 ---
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.metrics import mean_absolute_error

# -- CJK font setup (避免中文標籤顯示為方框) --
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["hospitalization_date"] = pd.to_datetime(df["hospitalization_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
cases = df[df["infected"] == 1]

# 每日發病數，補齊無發病的日期（確保連續）
daily = cases.groupby("symptom_onset_date").size()
daily = daily.asfreq("D", fill_value=0)
daily.name = "cases"
print(f"序列長度：{len(daily)} 天 | 總病例：{daily.sum()}")
print(daily.head(10))

In [ ]:
# --- Step 2: 流行曲線 + 7 日滾動平均 ---
rolling_7 = daily.rolling(window=7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, width=1.0,
       color="#6A9BCC", edgecolor="white", alpha=0.65, label="每日新增")
ax.plot(rolling_7.index, rolling_7.values, color="#D97757", linewidth=2,
        label="7 日滾動平均")
ax.set_title("松柏護理之家退伍軍人症流行曲線", fontweight="bold")
ax.set_xlabel("發病日期 (Date of Symptom Onset)")
ax.set_ylabel("病例數 (Number of Cases)")
ax.legend(); ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

## Part A ── 短期 outbreak 預測（護理之家資料）

17 天的 outbreak 資料足夠訓練 rolling mean / Poisson / NB / Logistic，但**不夠訓練 ARIMA / SARIMA**（那是 Part B 的任務）。

In [ ]:
# --- Step 3: Baseline — Rolling mean 預測 ---
# 用前 w 天的平均預測「下一天」，shift(1) 避免 data leakage
print("=== Rolling mean（不同窗口的 MAE）===")
mae_by_window = {}
for w in [3, 5, 7]:
    pred_w = daily.rolling(window=w).mean().shift(1).dropna()
    actual_w = daily.loc[pred_w.index]
    mae_by_window[w] = mean_absolute_error(actual_w, pred_w)
    print(f"  window={w}  MAE={mae_by_window[w]:.3f}")

mae_rolling = mae_by_window[3]
print(f"\n\u2192 最佳：window=3, MAE={mae_rolling:.3f}")

In [ ]:
# --- Step 4: Lagged features — 把「昨天、前天」變成特徵 ---
ts = daily.to_frame("cases").reset_index(names="date")
ts["day_idx"] = range(len(ts))       # 天數編號（趨勢）
ts["lag_1"] = ts["cases"].shift(1)   # 昨天的病例數
ts["lag_2"] = ts["cases"].shift(2)   # 前天的病例數

# 前兩列沒有「前兩天」可參考 → NaN → dropna 掉
ts_model = ts.dropna().reset_index(drop=True)
print(ts_model.head())
print(f"\n可用列數：{len(ts_model)}")

In [ ]:
# --- Step 5: Poisson regression + lag ---
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Poisson GLM：cases ~ 昨天 + 前天 + 天數趨勢
model_pois = smf.glm(
    "cases ~ lag_1 + lag_2 + day_idx",
    data=ts_model,
    family=sm.families.Poisson(),
).fit()

pred_pois = model_pois.predict(ts_model)
mae_pois = mean_absolute_error(ts_model["cases"], pred_pois)
print(f"Poisson + lag:  MAE={mae_pois:.3f},  AIC={model_pois.aic:.2f}")

# 解讀係數：exp(β) = incidence rate ratio (IRR)
coef_table = pd.DataFrame({
    "coef (log scale)": model_pois.params,
    "IRR exp(coef)": np.exp(model_pois.params),
})
print("\n=== 係數表 ===")
print(coef_table.round(3))

In [ ]:
# --- Step 6: Negative Binomial regression — 處理過度離散 ---
# 先檢查 dispersion ratio
disp = ts_model["cases"].var() / ts_model["cases"].mean()
print(f"dispersion = variance / mean = {disp:.2f}")
print("\u2192 > 1.5 視為過度離散 \u2192 改用 Negative Binomial\n")

# Negative Binomial GLM
model_nb = smf.glm(
    "cases ~ lag_1 + lag_2 + day_idx",
    data=ts_model,
    family=sm.families.NegativeBinomial(alpha=1.0),
).fit()

pred_nb = model_nb.predict(ts_model)
mae_nb = mean_absolute_error(ts_model["cases"], pred_nb)
print(f"Negative Binomial + lag:  MAE={mae_nb:.3f},  AIC={model_nb.aic:.2f}")

In [ ]:
# --- Step 7: Logistic regression — 「明天會不會是高峰日？」 ---
# 把 75th percentile 當「高峰日」門檻
threshold = ts_model["cases"].quantile(0.75)
ts_model["high_day"] = (ts_model["cases"] > threshold).astype(int)
print(f"高峰日門檻（>75th）= {threshold:.0f} 人")
print(f"高峰日數：{ts_model['high_day'].sum()} / {len(ts_model)} 天\n")

# 用昨天、前天的病例數預測「明天會不會超過門檻」
model_logit = smf.logit("high_day ~ lag_1 + lag_2", data=ts_model).fit(disp=False)
prob = model_logit.predict(ts_model)
pred_binary = (prob > 0.5).astype(int)
acc = (pred_binary == ts_model["high_day"]).mean()
print(f"Logistic (threshold): accuracy = {acc:.3f}")

print("\n=== 前 5 天的預測機率 ===")
demo = ts_model[["date", "cases", "lag_1", "lag_2", "high_day"]].copy()
demo["P(high)"] = prob.round(3)
print(demo.head())

## Part B ── 長期監測預測（合成 90 天類流感資料）

ARIMA / SARIMA 需要 ≥ 30 天（SARIMA 更需要至少 2 個完整週期）。Outbreak 資料只有 17 天，硬套會不穩。這裡我們**合成一條 90 天的類流感每日通報數**，包含趨勢 + 7 天週循環 + 噪音。

In [ ]:
# --- Step 8: 合成 90 天監測序列（trend + 7 天週期 + 噪音）---
rng = np.random.default_rng(42)
n_days = 90
dates = pd.date_range("2025-10-01", periods=n_days, freq="D")

trend = np.linspace(3, 7, n_days)                          # 長期上升趨勢
seasonal = 3 * np.sin(2 * np.pi * np.arange(n_days) / 7)   # 7 天週期
noise = rng.normal(0, 1.2, n_days)                          # 隨機噪音
synth_cases = np.maximum(0, (trend + seasonal + noise).round()).astype(int)
synth = pd.Series(synth_cases, index=dates, name="cases")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(synth.index, synth.values, color="#6A9BCC", linewidth=1.5)
ax.set_title("合成類流感每日通報數（trend + 7 天週期 + 噪音）", fontweight="bold")
ax.set_xlabel("日期"); ax.set_ylabel("每日通報數")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

print(f"序列長度：{len(synth)} 天 | 均值：{synth.mean():.2f} | 變異：{synth.var():.2f}")

In [ ]:
# --- Step 9: ARIMA(1, 1, 1) ---
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

# 平穩性檢定
adf_stat, p_value, *_ = adfuller(synth)
print(f"ADF statistic = {adf_stat:.3f}, p-value = {p_value:.3f}")
print("\u2192 p < 0.05 表示序列平穩；否則 d \u2265 1\n")

# 切訓練 / 測試集：前 83 天訓練，最後 7 天測試
train, test = synth.iloc[:-7], synth.iloc[-7:]

model_arima = ARIMA(train, order=(1, 1, 1)).fit()
forecast_arima = model_arima.forecast(steps=7)
mae_arima = mean_absolute_error(test.values, forecast_arima.values)
print(f"ARIMA(1,1,1):  MAE={mae_arima:.3f},  AIC={model_arima.aic:.2f}")

In [ ]:
# --- Step 10: SARIMA(1,1,1)(1,1,1,7) — 加入週期性 ---
from statsmodels.tsa.statespace.sarimax import SARIMAX

model_sarima = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
).fit(disp=False)
forecast_sarima = model_sarima.forecast(steps=7)
mae_sarima = mean_absolute_error(test.values, forecast_sarima.values)
print(f"SARIMA(1,1,1)(1,1,1,7):  MAE={mae_sarima:.3f},  AIC={model_sarima.aic:.2f}")

# 視覺化：ARIMA vs SARIMA vs 實際
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train.index[-30:], train.values[-30:], color="#6B6B6B",
        linewidth=1.2, label="訓練（最後 30 天）")
ax.plot(test.index, test.values, color="#1A1A1A", linewidth=2,
        marker="o", markersize=5, label="實際")
ax.plot(test.index, forecast_arima.values, color="#6A9BCC", linewidth=1.8,
        marker="s", markersize=5, linestyle="--", label=f"ARIMA (MAE={mae_arima:.2f})")
ax.plot(test.index, forecast_sarima.values, color="#D97757", linewidth=1.8,
        marker="^", markersize=5, linestyle="--", label=f"SARIMA (MAE={mae_sarima:.2f})")
ax.set_title("ARIMA vs SARIMA 未來 7 天預測", fontweight="bold")
ax.set_xlabel("日期"); ax.set_ylabel("每日通報數")
ax.legend(loc="upper left"); ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

In [ ]:
# --- Step 11: 模型大比拼 ---
comparison = pd.DataFrame([
    {"model": "\u2460 Rolling mean (w=3)",      "資料集": "outbreak",  "MAE": f"{mae_rolling:.3f}",
     "最少資料": "5 天",  "捕捉週期": "否",    "信賴區間": "否"},
    {"model": "\u2461 Poisson + lag",           "資料集": "outbreak",  "MAE": f"{mae_pois:.3f}",
     "最少資料": "10 天", "捕捉週期": "部分",  "信賴區間": "是"},
    {"model": "\u2462 Negative Binomial + lag", "資料集": "outbreak",  "MAE": f"{mae_nb:.3f}",
     "最少資料": "10 天", "捕捉週期": "部分",  "信賴區間": "是"},
    {"model": "\u2463 Logistic (threshold)",    "資料集": "outbreak",  "MAE": f"— (acc={acc:.2f})",
     "最少資料": "10 天", "捕捉週期": "否",    "信賴區間": "是（機率）"},
    {"model": "\u2464 ARIMA(1,1,1)",            "資料集": "synth 90d", "MAE": f"{mae_arima:.3f}",
     "最少資料": "30 天", "捕捉週期": "弱",    "信賴區間": "是"},
    {"model": "\u2465 SARIMA(1,1,1)(1,1,1,7)",  "資料集": "synth 90d", "MAE": f"{mae_sarima:.3f}",
     "最少資料": "60 天", "捕捉週期": "強",    "信賴區間": "是"},
])
print(comparison.to_string(index=False))

In [ ]:
# --- Step 12: 發病 vs 住院曲線（Lag 效應）---
hosp_daily = (
    cases[cases["hospitalization_date"].notna()]
    .groupby("hospitalization_date").size()
)
all_dates = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
hosp_aligned = hosp_daily.reindex(all_dates, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, width=1.0, alpha=0.55,
       color="#6A9BCC", edgecolor="white", label="發病")
ax.bar(hosp_aligned.index, hosp_aligned.values, width=1.0, alpha=0.55,
       color="#D97757", edgecolor="white", label="住院")
ax.set_title("發病 vs 住院曲線（Lag 效應）", fontweight="bold")
ax.set_xlabel("日期 (Date)"); ax.set_ylabel("人數 (Number of Cases)")
ax.legend(); ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

lag_days = (hosp_aligned.idxmax() - daily.idxmax()).days
print(f"發病高峰 {daily.idxmax().date()} \u2192 住院高峰 {hosp_aligned.idxmax().date()}")
print(f"Lag = {lag_days} 天 \u2192 可提前 {lag_days} 天調度床位與醫療人力")

## 小結

| 模型 | 適合情境 | 核心指令 |
|------|----------|----------|
| ① Rolling mean | outbreak 剛爆發、資料極少 | `daily.rolling(w).mean().shift(1)` |
| ② Poisson + lag | 計數資料 + 想放解釋變項 | `smf.glm(..., family=Poisson())` |
| ③ Negative Binomial | variance ≫ mean（群聚） | `sm.families.NegativeBinomial(alpha=1.0)` |
| ④ Logistic (threshold) | 是/否警報 | `smf.logit('high_day ~ lag_1 + lag_2')` |
| ⑤ ARIMA | ≥ 30 天、無明顯週期 | `ARIMA(y, order=(p,d,q)).fit()` |
| ⑥ SARIMA | ≥ 60 天、有週期 | `SARIMAX(y, seasonal_order=(P,D,Q,s))` |

**結論**：沒有「最佳模型」——選哪個要看資料長度、是否有週期、以及你要預測**連續數字**還是**是/否警報**。

下一章（Ch08），我們問「在哪裡」最嚴重？→ 空間流病。